In [3]:
#Assignment 5: Gabriel Maldonado

In [1]:
import pygame
import moderngl
import numpy
import math
import glm

pygame.init()

width = 600
height = 600
pygame.display.gl_set_attribute(pygame.GL_CONTEXT_PROFILE_MASK, pygame.GL_CONTEXT_PROFILE_CORE)
display = pygame.display.set_mode((width, height), flags= pygame.OPENGL | pygame.DOUBLEBUF | pygame.RESIZABLE)
display = pygame.display.set_caption(title = "Assignment 5 - Gabriel Maldonado")
gl = moderngl.get_context()
gl.info["GL_VERSION"]

shape_position_data = [
    0.0, 0.8,
    -0.8, -0.0,
    0.8, 0.0,
    0.8, -0.0,
    -0.8, 0.0,
    0.0, -0.8
]

texture_position_data = [
    -1.0, -1.0,
    1.0, -1.0,
    -1.0, 1.0,
     1.0, -1.0,
    1.0, 1.0,
    -1.0, 1.0
]
texture_coords = [
    0.0, 0.0,
    1.0, 0.0,
    0.0, 1.0,
    1.0, 0.0,
    1.0, 1.0,
    0.0, 1.0
]

shape_geometry = numpy.array(shape_position_data).astype("float32")
texture_geometry = numpy.array(texture_coords).astype("float32")
texture_positions = numpy.array(texture_position_data).astype("float32")

shape_geometry_buffer = gl.buffer(shape_geometry)
texture_geometry_buffer = gl.buffer(texture_geometry)
texture_positions_buffer = gl.buffer(texture_positions)

shape_vertex_shader_code = '''
    #version 330 core

    layout (location=0);
    in vec2 positions;
    
    uniform float scale;
    uniform float radius;
    uniform mat2 matrix;
    
    void main(){
        vec2 displacement = radius* matrix * vec2(0,1);
        vec2 P = matrix * (scale * positions) + displacement;

        gl_Position = vec4(P, 0.0, 1.0);
    }
    '''

line_vertex_shader_code = '''
    #version 330 core

    uniform float radius;
    uniform mat2 matrix;
    
    void main(){
        vec2 P = vec2(0);
        if (gl_VertexID > 0){
            P = radius * matrix * vec2(0,1);
        }
        gl_Position = vec4(P, 0.0, 1.0);
    }
'''

texture_vertex_shader_code = '''
    #version 330 core

    layout(location=0);
    in vec2 positions;
    in vec2 inTexCoords;
    out vec2 texCoords;
    uniform mat2 aspectMatrix;

    void main(){ 
        vec2 displacement = aspectMatrix * vec2(0,0);
        vec2 P = aspectMatrix * positions + displacement;
        gl_Position = vec4(P,0,1);
        texCoords = inTexCoords;
}

'''

fragment_shader_code = '''
    #version 330 core
    

    out vec4 outColor;
    uniform vec3 inColor;
    void main(){
        //set output variable value here
        outColor = vec4(inColor,1);        
    }
    '''

texture_fragment_shader_code = '''
    #version 330 core

    in vec2 texCoords;
    uniform sampler2D tex;
    out vec4 tColor;

    void main(){
        tColor = texture(tex, texCoords);
}    
'''

ctx = moderngl.create_context()

shapeProgram = gl.program(
    vertex_shader = shape_vertex_shader_code,
    fragment_shader = fragment_shader_code,
)
lineProgram = gl.program(
    vertex_shader = line_vertex_shader_code,
    fragment_shader = fragment_shader_code
)
textureProgram = gl.program(
    vertex_shader = texture_vertex_shader_code,
    fragment_shader = texture_fragment_shader_code
)


shapeRenderable = gl.vertex_array(shapeProgram,
                                  [(shape_geometry_buffer, "2f", "positions")]
)
lineRenderable = gl.vertex_array(lineProgram,
                                 []
)
textureRenderable = gl.vertex_array(textureProgram, [
    (texture_positions_buffer, "2f", "positions"),
    (texture_geometry_buffer, "2f", "inTexCoords")]
)

texture_img = pygame.image.load("clockFace.png")
texture_data = pygame.image.tobytes(texture_img, "RGBA", True)
texture_width = texture_img.get_width()
texture_height = texture_img.get_height()
texture_dim = (texture_width, texture_height)

texture = ctx.texture(texture_dim, 4, texture_data)
texture.build_mipmaps
repeat_x = True
repeat_y = True
sampler = ctx.sampler(texture = texture, repeat_x=True, repeat_y=True, filter = (gl.LINEAR, gl.LINEAR))
sampler.texture = texture



running = True
clock = pygame.time.Clock()
angle = 0

while running:
    transformMatrix = glm.scale(glm.mat4(1), glm.vec3(1,1,1))
    for event in pygame.event.get():
        if event.type == pygame.QUIT or (event.type == pygame.KEYDOWN and event.key == 27):
            running = False
        elif (event.type == pygame.WINDOWRESIZED):
            width = event.x
            height = event.y
    aspect = width/height
    M_aspect = glm.mat2(1/aspect, 0, 0, 1) if aspect > 1 else glm.mat2(1, 0, 0, aspect)

    cos = math.cos(math.radians(angle))
    sin = math.sin(math.radians(angle))
    M_rotate = glm.mat2(cos, -sin, sin, cos)
    
    M = M_aspect * M_rotate        
                
            
    gl.clear(0.5,0.5,0.0)

    sampler.use(0)
    textureProgram["tex"] = 0
    textureProgram["aspectMatrix"].write(M_aspect)
    textureRenderable.render()

    shapeProgram["matrix"].write(M)
            
    shapeProgram["inColor"] = [1,1,0]
    shapeProgram["scale"] = 0.1
    
    shapeProgram["radius"] = 0.8
    shapeRenderable.render()

    shapeProgram["radius"] = 0.0
    shapeRenderable.render()

    lineProgram["matrix"].write(M)
    lineProgram["inColor"] = [0,0,0]
    lineProgram["radius"] = 0.8
    lineRenderable.render(moderngl.LINES, vertices=2)
    
    pygame.display.flip()
    clock.tick(10)
    angle = angle + 1
    if (angle > 360):
        angle = 0    
    
pygame.display.quit()


C:\Users\vanes\AppData\Roaming\Python\Python313\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.7)
Hello from the pygame community. https://www.pygame.org/contribute.html
